# Week 11 -- Model Explainability & Fraud Decision Insights

**FraudFlow Project | Week 11**

Understanding the fraud prediction model using SHAP (SHapley Additive exPlanations). The explainability layer answers which features drive the model globally, and exactly why individual transactions were flagged or approved at the decision threshold.


## 1. Objective

- Implement production-oriented explainability for the final XGBoost model.
- Calculate **Global Feature Importance** using mean absolute SHAP values.
- Provide **Local Explanations** for High-Risk, Low-Risk, and Borderline decisions.
- Mathematically verify the SHAP consistency against the model's output margin.
- Preserve data leakage safeguards (no target leakage in explanations).


## 2. Load Existing Model and Configuration

We load the Week 8 XGBoost artifact and the Week 10 optimal threshold.


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.special import expit
from src.preprocessing.feature_engineering import engineer_features

with open('reports/phase10_optimization_metrics.json') as f:
    w10 = json.load(f)
THRESHOLD = w10['selected_model']['optimal_threshold']
print(f'Production Decision Threshold: {THRESHOLD}')

art = joblib.load('models/xgboost_week8.joblib')
model = art['model']
preprocessor = art['preprocessor']
print('Model loaded successfully.')

## 3. Data Preparation

We extract a random sample from the test set for explanations. Test data can safely be used here because model training and threshold tuning are strictly frozen.


In [ ]:
df_raw = pd.read_csv('data/raw/fraudTest.csv')
df = engineer_features(df_raw)

y = df['is_fraud']
X = df.drop(columns=['is_fraud', 'fraud_loss_amount'], errors='ignore')

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_test_proc = preprocessor.transform(X_test)
if hasattr(X_test_proc, 'toarray'):
    X_test_proc = X_test_proc.toarray().astype('float32')

num_cols = preprocessor.transformers_[0][2]
cat_enc  = preprocessor.named_transformers_['cat']
cat_cols = preprocessor.transformers_[1][2]
cat_names = cat_enc.get_feature_names_out(cat_cols)
feature_names = [f.replace('[', '(').replace(']', ')').replace('<', 'lt_') for f in list(num_cols) + list(cat_names)]

# 5000-row sample for computational efficiency
np.random.seed(42)
sample_idx = np.random.choice(len(y_test), min(5000, len(y_test)), replace=False)
X_shap = X_test_proc[sample_idx]
df_shap = pd.DataFrame(X_shap, columns=feature_names)
print(f'Sample shape for SHAP: {df_shap.shape}')

## 4. Compute SHAP Values

XGBoost predictions are interpreted natively in Log-Odds space using `shap.TreeExplainer`.


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(df_shap)
base_value = explainer.expected_value
if isinstance(base_value, np.ndarray): base_value = base_value[0]
print(f'Base value (log-odds expected value over background): {base_value:.4f}')

## 5. SHAP Consistency Validation

Does `Base Value + Sum(SHAP Values)` equal the Model Output?


In [ ]:
margins = model.predict(X_shap, output_margin=True)
calculated = base_value + shap_values.sum(axis=1)
max_diff = np.abs(margins - calculated).max()
print(f'Maximum mathematical difference: {max_diff:.2e}')
assert max_diff < 1e-4, 'SHAP is inconsistent!'

## 6. Global Feature Importance

Which features contribute most strongly across all transactions?


In [ ]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
global_importance = pd.DataFrame({'Feature': feature_names, 'Mean |SHAP|': mean_abs_shap})
global_importance = global_importance.sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
display(global_importance.head(10))

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, df_shap, show=False)
plt.title('SHAP Summary (Beeswarm)')
plt.show()

## 7. Local Explainability: High-Risk Transaction

Understanding the model's decision for the highest-probability transaction in the sample.


In [ ]:
probs = expit(margins)
high_idx = np.argmax(probs)
print(f'--- High-Risk Transaction ---')
print(f'Predicted Probability: {probs[high_idx]:.4f}')
print(f'Decision (Threshold={THRESHOLD}): {"FRAUD" if probs[high_idx] >= THRESHOLD else "LEGIT"}')
shap.force_plot(base_value, shap_values[high_idx], df_shap.iloc[high_idx], matplotlib=True)

## 8. Local Explainability: Borderline Transaction

Explaining a difficult transaction precisely on the threshold boundary.


In [ ]:
border_idx = np.argmin(np.abs(probs - THRESHOLD))
print(f'--- Borderline Transaction ---')
print(f'Predicted Probability: {probs[border_idx]:.4f}')
print(f'Decision (Threshold={THRESHOLD}): {"FRAUD" if probs[border_idx] >= THRESHOLD else "LEGIT"}')
shap.force_plot(base_value, shap_values[border_idx], df_shap.iloc[border_idx], matplotlib=True)

## 9. Limitations and Interpretability Caveats

1. **Not Causal**: SHAP explains model behavior, not real-world causation. If `merchant_distance` has a high SHAP value, it means the model heavily relies on it, not that distance itself *causes* fraud.
2. **Feature Correlation**: If two features are highly correlated, SHAP may arbitrarily split importance between them.
3. **Local Scope**: A feature might be incredibly important globally, but irrelevant for a specific individual transaction (or vice-versa).
4. **No Target Leakage**: All explanations were derived purely from the trained mathematical function `f(x)`. Target labels were never supplied to the Explainer.
